In [68]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql

In [52]:
mistral = spark.read.parquet("CU2/encoder_baseline/MISTRAL/v1_das_mdp")
parhaf = spark.read.parquet("CU2/encoder_baseline/PARHAF")
syn = spark.read.parquet("CU2/encoder_baseline/syn")
mistral_test = spark.read.parquet("CU2/encoder_baseline/MISTRAL/test_v1_das")

#import pandas as pd
#parhaf = pd.read_parquet("/data/scratch/ldedieu/CU2/encoder_baseline/PARHAF")

In [53]:
#parhaf.head(5)

In [54]:
syn.show(5)

+------------+--------------------+----+
|     note_id|           note_text|  dp|
+------------+--------------------+----+
| 60129542198|enterite diarrhei...|A048|
|128849018914|enterite aigue, g...|A071|
|266287972385|tuberculome menin...|A171|
|352187318273|tuberculose, gang...|A183|
|455266533383|fievre causee par...|A259|
+------------+--------------------+----+
only showing top 5 rows



In [55]:
mistral.show(5)

+-----------+--------------------+-----+--------------------+----+
|    note_id|           note_text|   dp|                 das| mdp|
+-----------+--------------------+-----+--------------------+----+
|34359738370|Service d’Urgence...| O730| [F10202, I10, J982]|Z769|
|42949672963|Hôpital Michallon...|M7227|[Y839, M198, M353...|Z769|
|51539607555|Service de Néphro...| N185|[F1725, Z466, R41...|Z769|
|60129542146|SERVICE DE PÉDIAT...| R600|[A488, Q059, F711...|Z769|
|68719476738|Hôpital Cochin - ...|M4636|       [Z952, Z5910]|Z769|
+-----------+--------------------+-----+--------------------+----+
only showing top 5 rows



In [56]:
parhaf.show(5)

+-------------+--------------------+----+
|      note_id|           note_text|  dp|
+-------------+--------------------+----+
| 111669149696|Compte rendu de c...|S643|
| 111669149697|Compte rendu opér...|S643|
| 111669149698|Compte rendu d'ho...|S643|
|1305670057984|Compte rendu de c...|M171|
|1305670057985|Compte rendu opér...|M171|
+-------------+--------------------+----+
only showing top 5 rows



In [57]:
mistral_test.show(5)

+-----------+--------------------+-----+--------------------+----+
|    note_id|           note_text|   dp|                 das| mdp|
+-----------+--------------------+-----+--------------------+----+
| 8589934592|SERVICE DE CHIRUR...| I839|         [K743, I10]|Z769|
|17179869184|Service de Chirur...| K409|[E890, Z991, M109...|Z769|
|25769803776|SERVICE D'ORTHOPÉ...| S934|[Z993, S5250, E78...|Z769|
|34359738368|SERVICE DE NEURO-...|S0650|              [D638]|Z769|
|42949672960|Patient : Alain B...| C771|[Z538, C778, D611...|Z769|
+-----------+--------------------+-----+--------------------+----+
only showing top 5 rows



In [69]:
codes = pd.read_csv(
    "../data/Referentiel_CIM-10-20250108.csv", sep=";", header=1, dtype=str
)

codes = codes[["code", "libellé long", "code MCO/HAD"]].rename(
    columns={"libellé long": "definition", "code MCO/HAD": "mco_had"}
)
codes["code"] = codes["code"].astype(str).str.strip()
codes["mco_had"] = pd.to_numeric(codes["mco_had"], errors="coerce")

dp = codes[codes["mco_had"].isin([0])].drop("mco_had", axis=1)
das = codes[codes["mco_had"].isin([0,1,2])].drop("mco_had", axis=1)

dp=set(dp["code"].to_list())
das=set(das["code"].to_list())

In [58]:
labels_mistral = mistral.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
labels_mistral_test = mistral_test.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
labels_parhaf = parhaf.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
labels_syn = syn.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
print(len(labels_mistral))
print(len(labels_mistral_test))
print(len(labels_parhaf))
print(len(labels_syn))

[Stage 76:===================================================>(1979 + 5) / 2000]

1925
854
129
13328


In [70]:
unique_to_mistral = list(set(labels_mistral) - set(labels_syn)- set(labels_mistral_test))#- set(labels_parhaf)
print(len(unique_to_mistral))
unique_to_mistral

3


['U0715', 'U0711', 'Z52804']

In [71]:
bad = ["C09", "C03", "T918", "E10", "C50", "C05", "C10", "E11ni", "C67", "C04"]
mistral = mistral.filter(~F.col("dp").isin(bad))

In [72]:
print(len(labels_syn))

13328


In [73]:
print(len(labels_parhaf))

129


In [74]:
print(len(labels_mistral))

1925


In [75]:
unique_to_parhaf = list(set(labels_parhaf)- set(labels_syn))
print(len(unique_to_parhaf))
unique_to_parhaf

0


[]

In [77]:
unique_to_syn = list(set(labels_syn)- set(labels_mistral))
print(len(unique_to_syn))


11409


In [16]:
syn = syn.withColumn("flag", F.lit("syn"))
real=mistral.withColumn("flag", F.lit("real")).drop("mdp","das")

In [16]:
df = mistral

#df = real.unionByName(syn).drop(F.col("note_id"))
#print(df.count())
#assert(df.filter(F.col("flag")=="syn").count()==syn.count())
#assert(df.filter(F.col("flag")=="real").count()==real.count())


In [38]:
#df.show(5)

+--------------------+-----+----+
|           note_text|   dp|flag|
+--------------------+-----+----+
|Service d’Urgence...| O730|real|
|Hôpital Michallon...|M7227|real|
|Service de Néphro...| N185|real|
|SERVICE DE PÉDIAT...| R600|real|
|Hôpital Cochin - ...|M4636|real|
+--------------------+-----+----+
only showing top 5 rows



In [22]:
labels_dp = df.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
#labels_dp = parhaf.select("dp").distinct().rdd.flatMap(lambda x: x).collect()

print(len(labels_dp))

[Stage 31:=================================================> (1949 + 20) / 2000]

1925


In [18]:
# ====== DAS (multi-label) — counts = document-frequency per code ======
# weights from TRAIN only (mistral); das/mdp don't exist in syn

test = mistral_test


das_counts = (
    mistral.select(F.explode("das").alias("das"))   # one row per (doc, das code)
    .groupBy("das").agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)
das_freq_train = {r["das"]: r["count"] for r in das_counts.collect()}

# restrict label space by min frequency (plan §5: keep the head tractable)
MIN_DAS_FREQ = 0
label_freq_dict_das = {c: n for c, n in das_freq_train.items() if n >= MIN_DAS_FREQ}

# label space = train(kept) ∪ val, so PARHAF codes stay scoreable
das_val = test.select(F.explode("das").alias("das")).distinct().rdd.flatMap(lambda x: x).collect()
valid_labels_das = sorted(set(label_freq_dict_das) | (set(das_val) & set(das_freq_train)))
print(len(valid_labels_das), "DAS labels kept")

# ====== MDP (single-label, like DP) — Z769 already filled upstream ======
mdp_counts = (
    mistral.groupBy("mdp").agg(F.count("*").alias("count")).orderBy(F.desc("count"))
)
label_freq_dict_mdp = {r["mdp"]: r["count"] for r in mdp_counts.collect()}

mdp_val = test.select("mdp").distinct().rdd.flatMap(lambda x: x).collect()
valid_labels_mdp = sorted(set(label_freq_dict_mdp) | set(mdp_val))
print(len(valid_labels_mdp), "MDP labels (Z769 should dominate)")

4344 DAS labels kept


[Stage 29:=================================================> (1938 + 20) / 2000]

1 MDP labels (Z769 should dominate)


In [23]:
valid_labels_dp = list(labels_dp)
label_counts_dp = (
    df.groupBy("dp")
    #parhaf.groupBy("dp")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count")) 
)
label_freq_dict_dp = {
    row["dp"]: row["count"]
    for row in label_counts_dp.collect()
}

In [26]:
# create id2label and label2id:
head_names = ["dp", "das", "mdp"]
label2id = {head: {} for head in head_names}
id2label = {head: {} for head in head_names}
for head in head_names:
    if head=="dp":
        labels = valid_labels_dp 
    elif head=="das":
        labels = valid_labels_das
    elif head=="mdp":
        labels = valid_labels_mdp
    label2id[head] = {
        label: i for i, label in enumerate(labels)
    }
    id2label[head] = {
        i: label for i, label in enumerate(labels)
    }


In [20]:
#mistral_train = df.filter(F.col("flag")=="real")
#syn_train = df.filter(F.col("flag")=="syn")
#print(mistral_train.count())
#print(syn_train.count())

8586


[Stage 32:=============>                                         (30 + 0) / 122]

125138


In [21]:
from pyspark.sql import functions as F

def save_df_to_parquet(df, filename, num_partitions):
    df = df.withColumn("note_id", F.monotonically_increasing_id())

    df_to_save = df.select(
        "note_id",
        F.col("note_text"),
        F.col("dp")
    )

    df_to_save.repartition(num_partitions).write.mode("overwrite").parquet(filename)


# Save train/val/test as Parquet with ~1000 rows per file
save_df_to_parquet(mistral_train, "CU2/encoder_baseline/train/mistral", num_partitions=10)
save_df_to_parquet(syn_train, "CU2/encoder_baseline/train/syn", num_partitions=122)

In [22]:
import subprocess
user="ldedieu"
for split in ["mistral", "syn"]:
    subprocess.run(
        #["mkdir", "-p", f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2"],
        ["mkdir", "-p", f"/data/scratch/{user}/CU2/encoder_baseline/train/{split}"],
        check=True
    )

    subprocess.run(
        [
            "hdfs", "dfs", "-copyToLocal", "-f",
            #f"ia_codage/dataset_800k/{strat}_{n_class_dp}class_v2/*",
            #f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2/"
            f"CU2/encoder_baseline/train/{split}/*",
            f"/data/scratch/{user}/CU2/encoder_baseline/train/{split}/"
        ],
        check=True
    )

In [23]:
import pandas as pd
pd.to_pickle(valid_labels_dp, "../data/valid_labels_dp.pkl")
pd.to_pickle(label_freq_dict_dp,  "../data/label_freq_dict_dp.pkl")